# Unsupervised Clustering of Student Stress Patterns

This project examines patterns in student stress-related survey data using K-Means, Agglomerative Hierarchical Clustering, and DBSCAN. PCA, t-SNE, and SNE are used to examine the data in lower-dimensional spaces.

The dataset contains 1,100 records, 20 predictor variables, and the `stress_level` field. The target is excluded from preprocessing, dimensionality reduction, and model fitting; it is used only for post-clustering comparison.

**Dataset:** `StressLevelDataset.csv`  
**Research article:** [Comprehensive analysis of stress factors affecting students](https://doi.org/10.1007/s44163-024-00169-6)

The associated article was published in 2024. The CSV does not establish when the survey responses were collected, so the publication year is not treated as the collection year.

Keep `StressLevelDataset.csv` beside this notebook, or in a `data/` or `dataset/` subfolder. Run the cells in order.

## Analysis steps

1. Inspect the dataset and examine feature distributions.
2. Prepare and standardize the predictor variables.
3. Fit and compare three clustering algorithms.
4. Examine the data with PCA, t-SNE, and SNE.
5. Review cluster profiles and compare clusters with the held-out label.
6. Save figures, tables, and cluster assignments.

## 0. Environment and imports

In [ ]:
# Install these packages if they are not already available:
# %pip install numpy pandas matplotlib seaborn scikit-learn scipy

import sys
import json
import math
import warnings
import inspect
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.optimize import minimize
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score,
    pairwise_distances,
)
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN

warnings.filterwarnings('ignore', category=FutureWarning)
RANDOM_STATE = 42
TARGET = 'stress_level'

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (9, 5)

OUTPUT_DIR = Path('outputs_student_stress')
FIG_DIR = OUTPUT_DIR / 'figures'
TABLE_DIR = OUTPUT_DIR / 'tables'
MODEL_DIR = OUTPUT_DIR / 'models'
for folder in (OUTPUT_DIR, FIG_DIR, TABLE_DIR, MODEL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print('Python:', sys.version.split()[0])
print('Output folder:', OUTPUT_DIR.resolve())

## 1. Load the dataset

In [ ]:
possible_paths = [
    Path('StressLevelDataset.csv'),
    Path('data') / 'StressLevelDataset.csv',
    Path('dataset') / 'StressLevelDataset.csv',
]
DATA_PATH = next((path for path in possible_paths if path.exists()), None)

if DATA_PATH is None:
    matches = [
        path for path in Path.cwd().rglob('StressLevelDataset.csv')
        if not any(part.startswith('.') for part in path.parts)
        and 'site-packages' not in path.parts
    ]
    if matches:
        DATA_PATH = matches[0]

if DATA_PATH is None:
    raise FileNotFoundError(
        'Could not find StressLevelDataset.csv. Place it beside this notebook '
        'or inside a data/ or dataset/ folder, then rerun this cell.'
    )

df = pd.read_csv(DATA_PATH)
print('Loaded:', DATA_PATH.resolve())
print('Dataset shape:', df.shape)
display(df.head())

## 2. Dataset inspection and quality checks

In [ ]:
print('Dataset information:')
df.info()

print('\nDescriptive statistics:')
display(df.describe(include='all').T)

print('\nData types:')
display(df.dtypes.astype(str).rename('data_type').to_frame())

print('\nMissing values by column:')
display(df.isna().sum().rename('missing_count').to_frame())

print('Exact duplicate rows:', int(df.duplicated().sum()))
print('Target column exists:', TARGET in df.columns)

if TARGET not in df.columns:
    raise ValueError(f"Expected target column '{TARGET}' was not found. Check the CSV columns.")

if df.shape[0] < 10:
    raise ValueError('The dataset appears too small for the planned clustering experiments.')

# Summary of feature ranges and distinct values.
feature_summary = pd.DataFrame({
    'data_type': df.dtypes.astype(str),
    'unique_values': df.nunique(dropna=True),
    'minimum': df.min(numeric_only=True),
    'maximum': df.max(numeric_only=True),
})
display(feature_summary)
feature_summary.to_csv(TABLE_DIR / 'feature_summary.csv')

## 3. Exploratory data analysis (EDA)

Histograms, a Spearman correlation heatmap, and boxplots are used to examine feature distributions and relationships. Extreme values are reviewed but are not removed automatically, since they may represent valid survey responses.


In [ ]:
# Keep the target separate from the predictor variables.
X_raw = df.drop(columns=[TARGET]).copy()
y = df[TARGET].copy()

# Convert predictor columns to numeric values.
X_numeric = X_raw.apply(pd.to_numeric, errors='coerce')
non_numeric_after_conversion = X_numeric.columns[X_numeric.isna().all()].tolist()
if non_numeric_after_conversion:
    raise ValueError(
        'These predictor columns could not be converted to numeric values: '
        + ', '.join(non_numeric_after_conversion)
    )

print('Predictor shape:', X_numeric.shape)
print('Held-out target shape:', y.shape)
print('\nTarget value counts (descriptive only):')
display(y.value_counts(dropna=False).sort_index().rename('count').to_frame())

fig, ax = plt.subplots(figsize=(7, 4))
target_counts = y.value_counts(dropna=False).sort_index()
ax.bar(target_counts.index.astype(str), target_counts.values)
ax.set_title('Distribution of the Held-out Stress-Level Label')
ax.set_xlabel('Encoded stress_level value')
ax.set_ylabel('Number of records')
fig.tight_layout()
fig.savefig(FIG_DIR / 'target_distribution_posthoc_only.png', dpi=160, bbox_inches='tight')
plt.show()

n_cols = 4
n_rows = math.ceil(X_numeric.shape[1] / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = np.array(axes).reshape(-1)
for idx, column in enumerate(X_numeric.columns):
    axes[idx].hist(X_numeric[column].dropna(), bins='auto')
    axes[idx].set_title(column)
    axes[idx].set_xlabel('Encoded value')
for ax in axes[len(X_numeric.columns):]:
    ax.set_visible(False)
fig.suptitle('Distributions of Student Stress Predictor Features', y=1.01, fontsize=15)
fig.tight_layout()
fig.savefig(FIG_DIR / 'feature_distributions.png', dpi=160, bbox_inches='tight')
plt.show()

missing_total = int(X_numeric.isna().sum().sum())
if missing_total:
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.heatmap(X_numeric.isna(), cbar=False, cmap='magma', ax=ax)
    ax.set_title('Missing-Value Pattern in Predictor Features')
    fig.tight_layout()
    fig.savefig(FIG_DIR / 'missing_value_pattern.png', dpi=160, bbox_inches='tight')
    plt.show()
else:
    print('No missing values found in predictor features.')

# Spearman correlation between the ordinal-coded features.
correlation = X_numeric.corr(method='spearman')
fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(correlation, cmap='coolwarm', center=0, annot=True, fmt='.2f',
            linewidths=0.25, ax=ax)
ax.set_title('Spearman Correlation Between Predictor Features')
fig.tight_layout()
fig.savefig(FIG_DIR / 'spearman_correlation_heatmap.png', dpi=160, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(17, 8))
box_data = [X_numeric[column].dropna().to_numpy() for column in X_numeric.columns]
ax.boxplot(box_data, vert=False)
ax.set_yticks(range(1, len(X_numeric.columns) + 1), labels=X_numeric.columns)
ax.set_title('Boxplots of Predictor Features (Visual Inspection Only)')
ax.set_xlabel('Encoded value')
fig.tight_layout()
fig.savefig(FIG_DIR / 'feature_boxplots.png', dpi=160, bbox_inches='tight')
plt.show()

## 4. Preprocessing

The 20 predictor variables are stored as integer-coded scores or ratings. For the baseline distance-based analysis, their numeric ordering is retained. Missing values are median-imputed, and features are standardized so that variables with wider ranges do not dominate distance calculations.

The `stress_level` field is kept separate and is not used for imputation, scaling, PCA, SNE, t-SNE, or clustering.


In [ ]:
feature_names = X_numeric.columns.tolist()

# Median imputation.
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X_numeric)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

X_scaled_df = pd.DataFrame(X_scaled, columns=feature_names, index=df.index)
print('Processed feature matrix shape:', X_scaled.shape)
print('Any non-finite values after preprocessing:', not np.isfinite(X_scaled).all())
display(X_scaled_df.head())

# Save preprocessing statistics.
preprocessing_summary = pd.DataFrame({
    'feature': feature_names,
    'imputed_missing_count': X_numeric.isna().sum().values,
    'mean_after_scaling': X_scaled.mean(axis=0),
    'std_after_scaling': X_scaled.std(axis=0),
})
display(preprocessing_summary)
preprocessing_summary.to_csv(TABLE_DIR / 'preprocessing_summary.csv', index=False)

## 5. K-Means: selecting the number of clusters

In [ ]:
# Evaluate candidate cluster counts.
k_values = list(range(2, min(11, len(X_scaled))))
inertia_values = []
silhouette_values = []

for k in range(1, min(11, len(X_scaled))):
    model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    model.fit(X_scaled)
    inertia_values.append(model.inertia_)
    if k >= 2:
        silhouette_values.append(silhouette_score(X_scaled, model.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(range(1, len(inertia_values) + 1), inertia_values, marker='o')
axes[0].set_title('K-Means Elbow Method')
axes[0].set_xlabel('Number of clusters (K)')
axes[0].set_ylabel('Inertia')
axes[0].set_xticks(range(1, len(inertia_values) + 1))
axes[0].grid(True, alpha=0.3)
axes[1].plot(k_values, silhouette_values, marker='o')
axes[1].set_title('K-Means Silhouette Analysis')
axes[1].set_xlabel('Number of clusters (K)')
axes[1].set_ylabel('Silhouette score')
axes[1].set_xticks(k_values)
axes[1].grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / 'kmeans_elbow_and_silhouette.png', dpi=160, bbox_inches='tight')
plt.show()

k_selection = pd.DataFrame({
    'n_clusters': k_values,
    'inertia': inertia_values[1:],
    'silhouette_score': silhouette_values,
})
display(k_selection)
k_selection.to_csv(TABLE_DIR / 'kmeans_k_selection.csv', index=False)

# Select the initial K using the highest Silhouette score.
selected_k = int(k_selection.loc[k_selection['silhouette_score'].idxmax(), 'n_clusters'])
print('K selected by maximum observed Silhouette score:', selected_k)

kmeans_model = KMeans(n_clusters=selected_k, random_state=RANDOM_STATE, n_init=20)
kmeans_labels = kmeans_model.fit_predict(X_scaled)
print('\nK-Means cluster sizes:')
display(pd.Series(kmeans_labels, name='cluster').value_counts().sort_index().rename('records').to_frame())

## 6. Fit the clustering algorithms

K-Means and Agglomerative Clustering use the selected cluster count. DBSCAN is fitted after parameter tuning.

In [ ]:
# K-Means
cluster_labels = {'K-Means': kmeans_labels}

# Agglomerative Hierarchical Clustering
agg_model = AgglomerativeClustering(n_clusters=selected_k, linkage='ward')
agg_labels = agg_model.fit_predict(X_scaled)
cluster_labels['Agglomerative'] = agg_labels

print('Clustering algorithms:', list(cluster_labels.keys()))

## 7. DBSCAN parameter tuning

The parameter search considers both cluster separation and the proportion of observations assigned to clusters.

In [ ]:
# k-distance plot: the distance to the min_samples-th nearest neighbour helps inspect eps values.
min_samples_values = [5, 10, 15]
quantiles = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95, 0.98]
dbscan_trials = []
dbscan_label_cache = {}

# Plot a 5-nearest-neighbour distance curve.
knn = NearestNeighbors(n_neighbors=5)
knn.fit(X_scaled)
distances_5, _ = knn.kneighbors(X_scaled)
k_distance = np.sort(distances_5[:, -1])
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(k_distance)
ax.set_title('DBSCAN 5-Nearest-Neighbour Distance Plot')
ax.set_xlabel('Observations sorted by distance')
ax.set_ylabel('Distance to 5th nearest neighbour')
fig.tight_layout()
fig.savefig(FIG_DIR / 'dbscan_k_distance.png', dpi=160, bbox_inches='tight')
plt.show()

for min_samples in min_samples_values:
    nn = NearestNeighbors(n_neighbors=min_samples).fit(X_scaled)
    distances, _ = nn.kneighbors(X_scaled)
    kth_distances = distances[:, -1]
    eps_candidates = np.unique(np.round(np.quantile(kth_distances, quantiles), 3))

    for eps in eps_candidates:
        if eps <= 0:
            continue
        model = DBSCAN(eps=float(eps), min_samples=min_samples)
        labels = model.fit_predict(X_scaled)
        dbscan_label_cache[(float(eps), min_samples)] = labels

        non_noise = labels != -1
        n_clusters = len(set(labels[non_noise]))
        coverage = float(non_noise.mean())
        score = np.nan
        if n_clusters >= 2 and non_noise.sum() > n_clusters:
            try:
                score = silhouette_score(X_scaled[non_noise], labels[non_noise])
            except ValueError:
                pass

        dbscan_trials.append({
            'eps': float(eps),
            'min_samples': int(min_samples),
            'clusters_excluding_noise': int(n_clusters),
            'coverage_pct': coverage * 100,
            'noise_pct': (1 - coverage) * 100,
            'silhouette_non_noise': score,
        })

dbscan_tuning = pd.DataFrame(dbscan_trials).sort_values(
    ['coverage_pct', 'silhouette_non_noise'], ascending=[False, False]
).reset_index(drop=True)
display(dbscan_tuning)
dbscan_tuning.to_csv(TABLE_DIR / 'dbscan_tuning_results.csv', index=False)

# Selection rule: prefer configurations with >=2 clusters and >=50% coverage;
# within that set, maximize the non-noise Silhouette score. If none qualify,
# fall back to a valid multi-cluster setting with the highest coverage.
valid_dbscan = dbscan_tuning[
    (dbscan_tuning['clusters_excluding_noise'] >= 2)
    & dbscan_tuning['silhouette_non_noise'].notna()
]
preferred_dbscan = valid_dbscan[valid_dbscan['coverage_pct'] >= 50]

if not preferred_dbscan.empty:
    chosen_dbscan_row = preferred_dbscan.sort_values('silhouette_non_noise', ascending=False).iloc[0]
    dbscan_selection_note = 'Highest Silhouette score among configurations with at least 50% coverage and at least two clusters.'
elif not valid_dbscan.empty:
    chosen_dbscan_row = valid_dbscan.sort_values(
        ['coverage_pct', 'silhouette_non_noise'], ascending=[False, False]
    ).iloc[0]
    dbscan_selection_note = 'No multi-cluster configuration reached 50% coverage; selected the valid configuration with the highest coverage.'
else:
    chosen_dbscan_row = dbscan_tuning.sort_values('coverage_pct', ascending=False).iloc[0]
    dbscan_selection_note = 'No tested configuration produced two non-noise clusters; selected the configuration with the highest coverage, but DBSCAN separation metrics are not meaningful.'

dbscan_eps = float(chosen_dbscan_row['eps'])
dbscan_min_samples = int(chosen_dbscan_row['min_samples'])
db_labels = dbscan_label_cache[(dbscan_eps, dbscan_min_samples)]
cluster_labels['DBSCAN'] = db_labels

print(f'Selected DBSCAN eps={dbscan_eps:.3f}, min_samples={dbscan_min_samples}')
print(dbscan_selection_note)
print('DBSCAN clusters excluding noise:', len(set(db_labels) - {-1}))
print('DBSCAN noise percentage:', round((db_labels == -1).mean() * 100, 2))

## 8. Model evaluation

Internal metrics are calculated for K-Means, Agglomerative Clustering, and DBSCAN. DBSCAN noise observations are excluded from its internal scores.

In [ ]:
def clustering_metrics(labels, X_eval):
    """Return internal metrics; DBSCAN noise points (-1) are excluded."""
    labels = np.asarray(labels)
    mask = labels != -1
    labels_used = labels[mask]
    X_used = X_eval[mask]
    n_clusters = len(np.unique(labels_used))
    coverage = float(mask.mean())

    result = {
        'clusters': int(n_clusters),
        'coverage_pct': coverage * 100,
        'noise_pct': (1 - coverage) * 100,
        'silhouette': np.nan,
        'davies_bouldin': np.nan,
        'calinski_harabasz': np.nan,
    }
    if n_clusters >= 2 and len(labels_used) > n_clusters:
        result['silhouette'] = silhouette_score(X_used, labels_used)
        result['davies_bouldin'] = davies_bouldin_score(X_used, labels_used)
        result['calinski_harabasz'] = calinski_harabasz_score(X_used, labels_used)
    return result

metric_rows = []
for name, labels in cluster_labels.items():
    row = {'algorithm': name}
    row.update(clustering_metrics(labels, X_scaled))
    metric_rows.append(row)

model_comparison = pd.DataFrame(metric_rows)
display(model_comparison)
model_comparison.to_csv(TABLE_DIR / 'clustering_model_comparison.csv', index=False)

# Plot each evaluation metric separately.
for metric, title, ylabel in [
    ('silhouette', 'Silhouette Score Comparison', 'Silhouette score (higher is better)'),
    ('davies_bouldin', 'Davies-Bouldin Index Comparison', 'Davies-Bouldin index (lower is better)'),
    ('calinski_harabasz', 'Calinski-Harabasz Score Comparison', 'Calinski-Harabasz score (higher is better)'),
]:
    plot_data = model_comparison.dropna(subset=[metric])
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.bar(plot_data['algorithm'], plot_data[metric])
    ax.set_title(title)
    ax.set_xlabel('Algorithm')
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', rotation=20)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f'{metric}_comparison.png', dpi=160, bbox_inches='tight')
    plt.show()

print('Interpret DBSCAN metrics together with its coverage and noise percentage.')

## 9. Cluster sizes

In [ ]:
cluster_size_rows = []
for name, labels in cluster_labels.items():
    counts = pd.Series(labels).value_counts().sort_index()
    for cluster_id, count in counts.items():
        cluster_size_rows.append({
            'algorithm': name,
            'cluster_label': int(cluster_id),
            'records': int(count),
            'percentage': float(count / len(labels) * 100),
        })
cluster_sizes = pd.DataFrame(cluster_size_rows)
display(cluster_sizes)
cluster_sizes.to_csv(TABLE_DIR / 'cluster_sizes.csv', index=False)

fig, axes = plt.subplots(len(cluster_labels), 1, figsize=(10, 3.2 * len(cluster_labels)))
if len(cluster_labels) == 1:
    axes = [axes]
for ax, (name, labels) in zip(axes, cluster_labels.items()):
    counts = pd.Series(labels).value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts.values)
    ax.set_title(f'{name}: cluster distribution')
    ax.set_xlabel('Cluster label (-1 means DBSCAN noise)')
    ax.set_ylabel('Records')
fig.tight_layout()
fig.savefig(FIG_DIR / 'cluster_size_comparison.png', dpi=160, bbox_inches='tight')
plt.show()

## 10. Principal Component Analysis

In [ ]:
# Fit PCA to examine explained variance.
pca_full = PCA(random_state=RANDOM_STATE)
X_pca_full = pca_full.fit_transform(X_scaled)
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
n_components_95 = int(np.searchsorted(cumulative_variance, 0.95) + 1)

print('Components needed for at least 95% explained variance:', n_components_95)
print('Cumulative variance at that component:', round(cumulative_variance[n_components_95 - 1], 4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
            pca_full.explained_variance_ratio_)
axes[0].set_title('PCA Scree Plot')
axes[0].set_xlabel('Principal component')
axes[0].set_ylabel('Explained variance ratio')
axes[1].plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o')
axes[1].axhline(0.95, linestyle='--', label='95% variance')
axes[1].axvline(n_components_95, linestyle=':', label=f'{n_components_95} components')
axes[1].set_title('PCA Cumulative Explained Variance')
axes[1].set_xlabel('Number of components')
axes[1].set_ylabel('Cumulative explained variance')
axes[1].legend()
fig.tight_layout()
fig.savefig(FIG_DIR / 'pca_explained_variance.png', dpi=160, bbox_inches='tight')
plt.show()

pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca_2d = pca_2d.fit_transform(X_scaled)
pca_2d_variance = float(pca_2d.explained_variance_ratio_.sum())
print('Variance explained by PC1 and PC2:', round(pca_2d_variance, 4))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, (name, labels) in zip(axes, cluster_labels.items()):
    ax.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1], c=labels,
               cmap='tab10', s=24, alpha=0.8)
    ax.set_title(name)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
fig.suptitle('Clustering Results on the PCA Projection', fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / 'pca_cluster_comparison.png', dpi=160, bbox_inches='tight')
plt.show()

# PCA loadings for the first two components.
loadings = pd.DataFrame(
    pca_2d.components_.T,
    index=feature_names,
    columns=['PC1_loading', 'PC2_loading'],
)
display(loadings.sort_values('PC1_loading', key=np.abs, ascending=False))
loadings.to_csv(TABLE_DIR / 'pca_loadings_pc1_pc2.csv')

fig, ax = plt.subplots(figsize=(7, 9))
sns.heatmap(loadings, cmap='coolwarm', center=0, annot=True, fmt='.2f', ax=ax)
ax.set_title('PCA Loadings: First Two Components')
fig.tight_layout()
fig.savefig(FIG_DIR / 'pca_loadings_heatmap.png', dpi=160, bbox_inches='tight')
plt.show()

# Compare K-Means before and after retaining at least 95% of PCA variance.
pca_95 = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_pca_95 = pca_95.fit_transform(X_scaled)
print('PCA-95 feature matrix shape:', X_pca_95.shape)

kmeans_pca95 = KMeans(n_clusters=selected_k, random_state=RANDOM_STATE, n_init=20)
kmeans_pca95_labels = kmeans_pca95.fit_predict(X_pca_95)
pca95_metrics = clustering_metrics(kmeans_pca95_labels, X_pca_95)
original_metrics = clustering_metrics(kmeans_labels, X_scaled)
pca_space_comparison = pd.DataFrame([
    {'feature_space': 'Original standardized features', **original_metrics},
    {'feature_space': 'PCA space (>=95% variance)', **pca95_metrics},
])
display(pca_space_comparison)
pca_space_comparison.to_csv(TABLE_DIR / 'original_vs_pca95_kmeans.csv', index=False)

## 11. t-SNE visualization

t-SNE is used to visualize local neighbourhood structure in two dimensions. The embedding is not used as the primary space for evaluating clustering quality.

In [ ]:
# t-SNE provides a two-dimensional view of local neighbourhood structure.
tsne_parameters = inspect.signature(TSNE).parameters
iteration_argument = 'max_iter' if 'max_iter' in tsne_parameters else 'n_iter'
tsne_kwargs = {
    'n_components': 2,
    'perplexity': min(30, max(5, (len(X_scaled) - 1) // 3)),
    'learning_rate': 'auto',
    'init': 'pca',
    'random_state': RANDOM_STATE,
}
tsne_kwargs[iteration_argument] = 1000

tsne_model = TSNE(**tsne_kwargs)
X_tsne = tsne_model.fit_transform(X_scaled)
print('t-SNE embedding shape:', X_tsne.shape)
print('t-SNE parameters:', tsne_kwargs)
print('Final t-SNE KL divergence:', float(tsne_model.kl_divergence_))
print('All t-SNE coordinates finite:', bool(np.isfinite(X_tsne).all()))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, (name, labels) in zip(axes, cluster_labels.items()):
    ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels,
               cmap='tab10', s=22, alpha=0.8)
    ax.set_title(name)
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
fig.suptitle('Clustering Results on the t-SNE Projection', fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / 'tsne_cluster_comparison.png', dpi=160, bbox_inches='tight')
plt.show()

pd.DataFrame(X_tsne, columns=['tsne_1', 'tsne_2']).to_csv(
    TABLE_DIR / 'tsne_embedding.csv', index=False
)

## 12. SNE visualization

Original SNE uses conditional Gaussian probabilities. A fixed sample of up to 800 records is used to keep the calculation manageable.

In [ ]:
from scipy.special import logsumexp


def _entropy_and_probabilities(distances, beta):
    """Return entropy and conditional Gaussian probabilities for one row."""
    log_weights = -beta * distances
    log_normalizer = logsumexp(log_weights)
    probabilities = np.exp(log_weights - log_normalizer)
    entropy = log_normalizer + beta * np.sum(distances * probabilities)
    return entropy, probabilities


def compute_sne_conditional_probabilities(
    X_input, perplexity=30.0, tolerance=1e-5, max_tries=60
):
    """Compute original-SNE high-dimensional conditional probabilities p(j|i)."""
    n_samples = X_input.shape[0]
    if not 1 < perplexity < n_samples:
        raise ValueError('perplexity must be greater than 1 and smaller than the sample count')

    distances = pairwise_distances(X_input, metric='euclidean', squared=True)
    conditional = np.zeros((n_samples, n_samples), dtype=float)
    target_entropy = np.log(perplexity)

    for i in range(n_samples):
        mask = np.ones(n_samples, dtype=bool)
        mask[i] = False
        row_distances = distances[i, mask]

        beta = 1.0
        beta_min = 0.0
        beta_max = np.inf

        for _ in range(max_tries):
            entropy, row_probabilities = _entropy_and_probabilities(row_distances, beta)
            difference = entropy - target_entropy
            if abs(difference) <= tolerance:
                break

            if difference > 0:
                # Entropy is too high: increase beta to make the distribution narrower.
                beta_min = beta
                beta = beta * 2.0 if np.isinf(beta_max) else (beta + beta_max) / 2.0
            else:
                # Entropy is too low: decrease beta to make the distribution broader.
                beta_max = beta
                beta = beta / 2.0 if beta_min == 0.0 else (beta + beta_min) / 2.0

        conditional[i, mask] = row_probabilities

    return conditional


def original_sne_objective(flat_embedding, conditional_probabilities, n_samples, n_components=2):
    """Original SNE KL objective and analytic gradient for Gaussian q(j|i)."""
    embedding = flat_embedding.reshape(n_samples, n_components)

    distances = pairwise_distances(embedding, metric='euclidean', squared=True)
    np.fill_diagonal(distances, np.inf)

    # Low-dimensional Gaussian conditional probabilities q(j|i).
    log_q = -distances
    log_q -= logsumexp(log_q, axis=1, keepdims=True)
    q = np.exp(log_q)

    # Sum_i KL(P_i || Q_i); omit the diagonal, where both probabilities are zero.
    positive = conditional_probabilities > 0
    cost = np.sum(
        conditional_probabilities[positive]
        * (np.log(conditional_probabilities[positive]) - log_q[positive])
    )

    # The point appears in both a conditional row and a conditional column.
    differences = embedding[:, None, :] - embedding[None, :, :]
    residual = (conditional_probabilities - q) + (conditional_probabilities.T - q.T)
    gradient = 2.0 * np.sum(residual[:, :, None] * differences, axis=1)

    return float(cost), gradient.ravel()


SNE_MAX_SAMPLES = min(800, len(X_scaled))
rng = np.random.default_rng(RANDOM_STATE)
sne_indices = np.sort(rng.choice(len(X_scaled), size=SNE_MAX_SAMPLES, replace=False))
X_sne_input = X_scaled[sne_indices]
sne_labels_for_plot = kmeans_labels[sne_indices]

sne_perplexity = min(30.0, max(5.0, (SNE_MAX_SAMPLES - 1) / 3.0))
P_sne_conditional = compute_sne_conditional_probabilities(
    X_sne_input,
    perplexity=sne_perplexity
)

# PCA initialization is centered and scaled to avoid a nearly uniform initial q matrix.
sne_initial = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_sne_input)
sne_initial = (sne_initial - sne_initial.mean(axis=0)) / (sne_initial.std(axis=0) + 1e-12)

sne_result = minimize(
    original_sne_objective,
    sne_initial.ravel(),
    args=(P_sne_conditional, SNE_MAX_SAMPLES, 2),
    method='L-BFGS-B',
    jac=True,
    options={
        'maxiter': 1200,
        'ftol': 1e-9,
        'gtol': 1e-5,
        'maxls': 50,
        'maxcor': 10,
    },
)

X_sne = sne_result.x.reshape(SNE_MAX_SAMPLES, 2)
sne_gradient_norm = float(np.linalg.norm(sne_result.jac))
print('Original SNE completed.')
print('Converged:', bool(sne_result.success))
print('Optimizer message:', sne_result.message)
print('Iterations:', int(sne_result.nit))
print('Function evaluations:', int(sne_result.nfev))
print('Final KL objective:', float(sne_result.fun))
print('Final gradient norm:', sne_gradient_norm)
print('Records used for SNE:', SNE_MAX_SAMPLES)
print('All SNE coordinates finite:', bool(np.isfinite(X_sne).all()))

if not sne_result.success:
    warnings.warn(
        'Original SNE optimizer did not report convergence. Treat its embedding as exploratory '
        'and report the optimizer message and gradient norm; do not describe it as converged.'
    )

sne_labels = {name: np.asarray(labels)[sne_indices]
              for name, labels in cluster_labels.items()}

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, (name, labels) in zip(axes, sne_labels.items()):
    ax.scatter(X_sne[:, 0], X_sne[:, 1], c=labels,
               cmap='tab10', s=24, alpha=0.8)
    ax.set_title(name)
    ax.set_xlabel('SNE 1')
    ax.set_ylabel('SNE 2')
fig.suptitle('Clustering Results on the SNE Projection', fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / 'original_sne_cluster_comparison.png', dpi=160, bbox_inches='tight')
plt.show()

sne_embedding_table = pd.DataFrame({
    'original_row_index': sne_indices,
    'sne_1': X_sne[:, 0],
    'sne_2': X_sne[:, 1],
})
for name, labels in sne_labels.items():
    column = name.lower().replace(' ', '_').replace('-', '_') + '_cluster'
    sne_embedding_table[column] = labels
sne_embedding_table.to_csv(TABLE_DIR / 'original_sne_embedding.csv', index=False)

sne_diagnostics = pd.DataFrame([{
    'perplexity': sne_perplexity,
    'sample_size': SNE_MAX_SAMPLES,
    'converged': bool(sne_result.success),
    'optimizer_message': str(sne_result.message),
    'iterations': int(sne_result.nit),
    'function_evaluations': int(sne_result.nfev),
    'final_kl_objective': float(sne_result.fun),
    'gradient_norm': sne_gradient_norm,
}])
display(sne_diagnostics)
sne_diagnostics.to_csv(TABLE_DIR / 'original_sne_diagnostics.csv', index=False)


## 13. Cluster profiles and post-clustering validation

K-Means cluster profiles are summarized in the original feature units. The held-out `stress_level` field is compared with the clustering results only after model fitting.

In [ ]:
# K-Means profile summary in the original feature units.
profile_data = X_numeric.copy()
profile_data['kmeans_cluster'] = kmeans_labels
cluster_profile_mean = profile_data.groupby('kmeans_cluster').mean()
cluster_profile_median = profile_data.groupby('kmeans_cluster').median()

print('K-Means cluster feature means:')
display(cluster_profile_mean)
print('K-Means cluster feature medians:')
display(cluster_profile_median)
cluster_profile_mean.to_csv(TABLE_DIR / 'kmeans_cluster_profile_means.csv')
cluster_profile_median.to_csv(TABLE_DIR / 'kmeans_cluster_profile_medians.csv')

# Heatmap of cluster means, standardized column-wise for display only.
profile_z = (cluster_profile_mean - cluster_profile_mean.mean()) / cluster_profile_mean.std(ddof=0).replace(0, 1)
fig, ax = plt.subplots(figsize=(16, 4.5))
sns.heatmap(profile_z, cmap='coolwarm', center=0, annot=True, fmt='.2f', ax=ax)
ax.set_title('K-Means Cluster Profile Heatmap (Column-standardized Means)')
ax.set_xlabel('Feature')
ax.set_ylabel('K-Means cluster')
fig.tight_layout()
fig.savefig(FIG_DIR / 'kmeans_cluster_profile_heatmap.png', dpi=160, bbox_inches='tight')
plt.show()

# Compare the fitted clusters with the held-out target.
posthoc_table = pd.crosstab(
    pd.Series(kmeans_labels, name='kmeans_cluster'),
    y.rename('encoded_stress_level'),
)
posthoc_row_pct = posthoc_table.div(posthoc_table.sum(axis=1), axis=0) * 100
print('Post-hoc cross-tabulation (counts):')
display(posthoc_table)
print('Post-hoc row percentages:')
display(posthoc_row_pct.round(2))
posthoc_table.to_csv(TABLE_DIR / 'kmeans_cluster_vs_stress_level_counts.csv')
posthoc_row_pct.to_csv(TABLE_DIR / 'kmeans_cluster_vs_stress_level_row_pct.csv')

# Calculate external validation scores.
external_rows = []
for name, labels in cluster_labels.items():
    if len(np.unique(labels)) >= 2:
        external_rows.append({
            'algorithm': name,
            'adjusted_rand_index_vs_stress_level': adjusted_rand_score(y, labels),
            'normalized_mutual_info_vs_stress_level': normalized_mutual_info_score(y, labels),
        })
external_validation = pd.DataFrame(external_rows)
print('Post-hoc external validation (not used for training or model selection):')
display(external_validation)
external_validation.to_csv(TABLE_DIR / 'posthoc_external_validation.csv', index=False)

## 14. Algorithm agreement and DBSCAN noise analysis

Pairwise ARI compares the cluster assignments from each pair of algorithms. The DBSCAN non-noise subset is also examined to see how the methods group the same assigned observations.

In [ ]:
# Pairwise Adjusted Rand Index on the full dataset and DBSCAN non-noise subset.
non_noise_mask = db_labels != -1
pairwise_rows = []

for name_1, name_2 in combinations(cluster_labels.keys(), 2):
    labels_1 = np.asarray(cluster_labels[name_1])
    labels_2 = np.asarray(cluster_labels[name_2])
    pairwise_rows.append({
        'algorithm_1': name_1,
        'algorithm_2': name_2,
        'ari_full_dataset': adjusted_rand_score(labels_1, labels_2),
        'ari_dbscan_non_noise': adjusted_rand_score(
            labels_1[non_noise_mask], labels_2[non_noise_mask]
        ),
    })

pairwise_comparison = pd.DataFrame(pairwise_rows)
display(pairwise_comparison.round(4))
pairwise_comparison.to_csv(TABLE_DIR / 'pairwise_ari_comparison.csv', index=False)

# Heatmaps of pairwise agreement.
algorithm_names = list(cluster_labels.keys())
ari_full = pd.DataFrame(np.eye(len(algorithm_names)),
                        index=algorithm_names, columns=algorithm_names)
ari_non_noise = ari_full.copy()
for row in pairwise_rows:
    a, b = row['algorithm_1'], row['algorithm_2']
    ari_full.loc[a, b] = ari_full.loc[b, a] = row['ari_full_dataset']
    ari_non_noise.loc[a, b] = ari_non_noise.loc[b, a] = row['ari_dbscan_non_noise']

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
sns.heatmap(ari_full, annot=True, fmt='.3f', vmin=0, vmax=1,
            cmap='Blues', square=True, ax=axes[0])
axes[0].set_title('Pairwise ARI: Full Dataset')
sns.heatmap(ari_non_noise, annot=True, fmt='.3f', vmin=0, vmax=1,
            cmap='Blues', square=True, ax=axes[1])
axes[1].set_title('Pairwise ARI: DBSCAN Non-noise Subset')
fig.tight_layout()
fig.savefig(FIG_DIR / 'pairwise_ari_comparison.png', dpi=160, bbox_inches='tight')
plt.show()

# Compare all three algorithms on the same observations assigned by DBSCAN.
subset_rows = []
for name, labels in cluster_labels.items():
    labels_subset = np.asarray(labels)[non_noise_mask]
    subset_rows.append({
        'algorithm': name,
        **clustering_metrics(labels_subset, X_scaled[non_noise_mask]),
    })

subset_comparison = pd.DataFrame(subset_rows)
print('Comparison on DBSCAN non-noise observations:')
display(subset_comparison.round(4))
subset_comparison.to_csv(TABLE_DIR / 'dbscan_non_noise_subset_comparison.csv', index=False)

# Cross-tabulate DBSCAN noise against K-Means and Agglomerative clusters.
noise_status = np.where(db_labels == -1, 'Noise', 'Assigned')
noise_tables = {}
for name in ['K-Means', 'Agglomerative']:
    table = pd.crosstab(
        pd.Series(noise_status, name='DBSCAN_status'),
        pd.Series(cluster_labels[name], name=f'{name}_cluster'),
    )
    noise_tables[name] = table
    print()
    print(f'DBSCAN status by {name} cluster:')
    display(table)
    safe_name = name.lower().replace(' ', '_')
    table.to_csv(TABLE_DIR / f'dbscan_noise_by_{safe_name}.csv')

# Summarize the number of noise records in each reference partition.
noise_summary_rows = []
for name in ['K-Means', 'Agglomerative']:
    labels = np.asarray(cluster_labels[name])
    for cluster_id in np.unique(labels):
        mask = labels == cluster_id
        noise_count = int(np.sum((db_labels == -1) & mask))
        total_count = int(mask.sum())
        noise_summary_rows.append({
            'algorithm': name,
            'cluster': int(cluster_id),
            'records': total_count,
            'dbscan_noise_records': noise_count,
            'noise_share_within_cluster_pct': 100 * noise_count / total_count,
        })

noise_distribution = pd.DataFrame(noise_summary_rows)
display(noise_distribution.round(2))
noise_distribution.to_csv(TABLE_DIR / 'dbscan_noise_distribution.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for ax, name in zip(axes, ['K-Means', 'Agglomerative']):
    data = noise_distribution[noise_distribution['algorithm'] == name]
    ax.bar(data['cluster'].astype(str), data['dbscan_noise_records'])
    ax.set_title(f'DBSCAN Noise by {name} Cluster')
    ax.set_xlabel('Cluster')
    ax.set_ylabel('Noise records')
fig.tight_layout()
fig.savefig(FIG_DIR / 'dbscan_noise_distribution.png', dpi=160, bbox_inches='tight')
plt.show()

## 15. K-Means stability across random seeds

In [ ]:
# Pairwise Adjusted Rand Index across repeated K-Means runs.
stability_labels = []
for seed in range(10):
    model = KMeans(n_clusters=selected_k, random_state=seed, n_init=10)
    stability_labels.append(model.fit_predict(X_scaled))

stability_scores = []
for i in range(len(stability_labels)):
    for j in range(i + 1, len(stability_labels)):
        stability_scores.append(adjusted_rand_score(stability_labels[i], stability_labels[j]))

print('Mean pairwise K-Means stability ARI:', round(float(np.mean(stability_scores)), 4))
print('Minimum pairwise K-Means stability ARI:', round(float(np.min(stability_scores)), 4))
pd.DataFrame({'pairwise_ari': stability_scores}).to_csv(
    TABLE_DIR / 'kmeans_stability_pairwise_ari.csv', index=False
)

## 16. Save final results and project artefacts

In [ ]:
# Save cluster assignments in the original row order.
assignments = pd.DataFrame(index=df.index)
assignments['original_stress_level_posthoc_only'] = y
for name, labels in cluster_labels.items():
    safe_name = name.lower().replace(' ', '_').replace('-', '_')
    assignments[f'{safe_name}_cluster'] = labels
assignments['pca_1'] = X_pca_2d[:, 0]
assignments['pca_2'] = X_pca_2d[:, 1]
assignments['tsne_1'] = X_tsne[:, 0]
assignments['tsne_2'] = X_tsne[:, 1]
assignments.to_csv(OUTPUT_DIR / 'final_cluster_assignments.csv', index=False)

# Save final comparison tables.
model_comparison.to_csv(OUTPUT_DIR / 'final_algorithm_comparison.csv', index=False)
cluster_sizes.to_csv(OUTPUT_DIR / 'final_cluster_sizes.csv', index=False)
pairwise_comparison.to_csv(OUTPUT_DIR / 'final_pairwise_ari_comparison.csv', index=False)
subset_comparison.to_csv(OUTPUT_DIR / 'final_dbscan_non_noise_comparison.csv', index=False)

metadata = {
    'dataset_file': DATA_PATH.name,
    'dataset_shape': [int(df.shape[0]), int(df.shape[1])],
    'predictor_count': int(X_scaled.shape[1]),
    'target_excluded_from_model_inputs': TARGET,
    'selected_k': int(selected_k),
    'dbscan_eps': float(dbscan_eps),
    'dbscan_min_samples': int(dbscan_min_samples),
    'dbscan_selection_note': dbscan_selection_note,
    'algorithms': list(cluster_labels.keys()),
    'pca_components_for_95_percent_variance': int(n_components_95),
    'pca_first_two_components_variance': pca_2d_variance,
    'tsne_parameters': {k: (int(v) if isinstance(v, np.integer) else v) for k, v in tsne_kwargs.items()},
    'original_sne_sample_size': int(SNE_MAX_SAMPLES),
    'original_sne_converged': bool(sne_result.success),
    'original_sne_optimizer_message': str(sne_result.message),
    'original_sne_iterations': int(sne_result.nit),
    'original_sne_final_kl_objective': float(sne_result.fun),
    'original_sne_gradient_norm': float(np.linalg.norm(sne_result.jac)),
    'random_state': RANDOM_STATE,
}
with open(OUTPUT_DIR / 'model_metadata.json', 'w', encoding='utf-8') as file:
    json.dump(metadata, file, indent=2, default=str)

print('Saved outputs to:', OUTPUT_DIR.resolve())
print('Figures:', FIG_DIR.resolve())
print('Tables:', TABLE_DIR.resolve())
print('Assignments:', (OUTPUT_DIR / 'final_cluster_assignments.csv').resolve())
print('Metadata:', (OUTPUT_DIR / 'model_metadata.json').resolve())

print()
print('Final internal metric comparison:')
display(model_comparison)

print()
print('Final external validation:')
display(external_validation)

print()
print('Pairwise algorithm agreement:')
display(pairwise_comparison)

print()
print('DBSCAN non-noise subset comparison:')
display(subset_comparison)

## 16. Notes for interpretation

Cluster numbers are identifiers and do not represent stress categories by themselves. Confirm the meanings of `stress_level` values 0, 1, and 2 from the dataset documentation before assigning names to them.

Silhouette, Davies–Bouldin, and Calinski–Harabasz scores describe cluster geometry. DBSCAN scores exclude noise observations, so coverage and noise percentage should be considered when comparing results. PCA, t-SNE, and SNE are lower-dimensional representations and should not be treated as proof of global cluster separation.

SNE uses a reproducible sample of up to 800 records. The sample size and optimizer diagnostics are saved with the results. The associated research article's publication year is 2024; the original survey collection year is not established by the CSV.